To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>Equation: x^5 + 3x^4 - 10 = 3. So x^5 + 3x^4 - 13 =0. Solve for real roots? maybe numeric. Let's try approximate.

We can test integer roots: try x=1 => 1+3


Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "Solve x^5 + 3x^4 - 10 = 3." Wait maybe it's an equation: x^5 + 3x^4 - 10 = 3. The variable x unknown. Solve for x. We need to solve the equation:

x^


Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation: x^5 + 3x^4 - 10 = 3. Or maybe it's x^5 + 3x^4 - 10 = 3? That seems like a polynomial equation: x^5 + 3x^4 - 10


<a name="Data"></a>
### Data Prep

The `HuggingFaceH4/Multilingual-Thinking` dataset will be utilized as our example. This dataset, available on Hugging Face, contains reasoning chain-of-thought examples derived from user questions that have been translated from English into four other languages. It is also the same dataset referenced in OpenAI's [cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) for fine-tuning. The purpose of using this dataset is to enable the model to learn and develop reasoning capabilities in these four distinct languages.

In [8]:
def formatting_prompts_func(examples):
    texts = []
    for convo in examples["messages"]:
        # Konvertujemo thinking proces u pravilne tokene
        formatted_convo = []
        for msg in convo:
            if msg["role"] == "assistant" and "thinking" in msg:
                # Dodajemo thinking tokene
                for thought in msg["thinking"]:
                    formatted_convo.append({
                        "role": "assistant",
                        "content": f"<|channel|>{thought['channel']}<|message|>{thought['content']}"
                    })
            # Dodajemo finalni odgovor
            if msg["role"] == "assistant" and "content" in msg and isinstance(msg["content"], dict):
                formatted_convo.append({
                    "role": "assistant",
                    "content": f"<|channel|>{msg['content']['channel']}<|message|>{msg['content']['text']}"
                })
            else:
                formatted_convo.append(msg)

        text = tokenizer.apply_chat_template(
            formatted_convo,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

from datasets import load_dataset

dataset = load_dataset("Chovus/500_dataset", split="train")
dataset

harmony_dataset.jsonl:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--Chovus--500_dataset/snapshots/d7c4ca1cbbb44619356ce895a0dd7f830451c1fa/harmony_dataset.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0
ERROR:datasets.packaged_modules.json.json:Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--Chovus--500_dataset/snapshots/d7c4ca1cbbb44619356ce895a0dd7f830451c1fa/harmony_dataset.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0


DatasetGenerationError: An error occurred while generating the dataset

In [30]:
import json
from datasets import Dataset
import os

# Putanja do tvog dataset fajla
dataset_path = "/root/.cache/huggingface/hub/datasets--Chovus--500_dataset/snapshots/d7c4ca1cbbb44619356ce895a0dd7f830451c1fa/harmony_dataset.jsonl"

def load_and_fix_dataset(file_path):
    fixed_data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            try:
                data = json.loads(line.strip())

                if "messages" not in data or not isinstance(data["messages"], list):
                    continue

                fixed_messages = []
                valid_conversation = True

                for msg in data["messages"]:
                    if not isinstance(msg, dict) or "role" not in msg:
                        valid_conversation = False
                        break

                    role = msg["role"]

                    # FIX 1: Obavezno content polje
                    if "content" not in msg:
                        valid_conversation = False
                        break

                    content = msg["content"]

                    # Konvertuj content u string ako je objekat
                    if isinstance(content, dict):
                        if role == "assistant" and "text" in content:
                            content = content["text"]
                        else:
                            content = str(content)
                    elif not isinstance(content, str):
                        content = str(content)

                    # FIX 2: Ujednači thinking polje
                    thinking = msg.get("thinking")
                    if thinking is None:
                        thinking = []  # Uvek lista, čak i ako je prazna
                    elif not isinstance(thinking, list):
                        thinking = []  # Ako nije lista, postavi na praznu

                    # Proveri da li su svi thinking elementi validni
                    cleaned_thinking = []
                    if thinking:
                        for thought in thinking:
                            if isinstance(thought, dict) and "channel" in thought and "content" in thought:
                                cleaned_thinking.append({
                                    "channel": str(thought["channel"]),
                                    "content": str(thought["content"])
                                })

                    fixed_msg = {
                        "role": role,
                        "content": content,
                        "thinking": cleaned_thinking  # Uvek lista
                    }

                    fixed_messages.append(fixed_msg)

                # Proveri da li ima bar jedan user i jedan assistant
                has_user = any(msg["role"] == "user" for msg in fixed_messages)
                has_assistant = any(msg["role"] == "assistant" for msg in fixed_messages)

                if valid_conversation and has_user and has_assistant and len(fixed_messages) >= 2:
                    fixed_data.append({
                        "messages": fixed_messages
                    })

            except json.JSONDecodeError as e:
                print(f"Greška u liniji {line_num}: {e}")
                continue
            except Exception as e:
                print(f"Neočekivana greška u liniji {line_num}: {e}")
                continue

    return fixed_data

print("Učitavam i popravljam dataset...")
fixed_dataset = load_and_fix_dataset(dataset_path)

if fixed_dataset:
    print(f"Uspešno popravljeno {len(fixed_dataset)} primera")

    # Kreiraj HuggingFace Dataset
    dataset = Dataset.from_list(fixed_dataset)
    print(dataset)

    # Prikaži strukturu prvog primera
    print("\n--- Struktura popravljenog unosa ---")
    for i, msg in enumerate(dataset[0]["messages"]):
        print(f"Poruka {i}: role={msg['role']}, content_len={len(msg['content'])}, thinking_len={len(msg['thinking'])}")

else:
    print("Nije pronađen nijedan ispravan primer!")

Učitavam i popravljam dataset...
Uspešno popravljeno 2684 primera
Dataset({
    features: ['messages'],
    num_rows: 2684
})

--- Struktura popravljenog unosa ---
Poruka 0: role=developer, content_len=135, thinking_len=0
Poruka 1: role=user, content_len=272, thinking_len=0
Poruka 2: role=assistant, content_len=1247, thinking_len=5


In [16]:
# Prikaži prvih 5 primera detaljno
print("=== PRVIH 5 PRIMERA ===")
for i in range(min(5, len(dataset))):
    print(f"\n--- Primer {i+1} ---")
    for j, msg in enumerate(dataset[i]["messages"]):
        print(f"Poruka {j}: role={msg['role']}")
        print(f"Content: {msg['content'][:200]}...")  # Prvih 200 karaktera
        if msg['thinking']:
            print(f"Thinking ({len(msg['thinking'])} thoughts):")
            for k, thought in enumerate(msg['thinking']):
                print(f"  {k+1}. [{thought['channel']}] {thought['content'][:100]}...")
        print("-" * 50)

=== PRVIH 5 PRIMERA ===

--- Primer 1 ---
Poruka 0: role=developer
Content: Ti si pravni savetnik specijalizovan za nasledno pravo u Srbiji. Koristiš analitički pristup za rešavanje složenih naslednih slučajeva....
--------------------------------------------------
Poruka 1: role=user
Content: Bila sam suvozač u kolima koja je vozio komšija kada me je povezao do posla. On je tom prilikom izazvao sudar i ja sam povređena, da li imam pravo da tražim od osiguranja da mi nadoknadi štetu. Neko m...
--------------------------------------------------
Poruka 2: role=assistant
Content: Vi imate pravo na naknadu stete od osiguranja. Prema clanu 18. Zakona o obaveznom osiguranju u saobracaju, Vlasnik motornog vozila dužan je da zaključi ugovor o osiguranju od odgovornosti za štetu koj...
Thinking (5 thoughts):
  1. [analysis] Pregled pitanja: korisnica traži pravnu pomoć oko mogućnosti traženja odštete od osiguranja u slučaj...
  2. [commentary] Član 18 Zakona o obaveznom osiguranju u saobraćaju 

In [29]:
from google.colab import files
import json

# Sačuvaj ceo dataset kao JSON fajl
with open('chovus_full_dataset.json', 'w', encoding='utf-8') as f:
    for i, example in enumerate(dataset):
        json.dump(example, f, ensure_ascii=False)
        f.write('\n')
        if i % 100 == 0:  # Progress indicator
            print(f"Sačuvano {i+1} primera...")

print("Dataset sačuvan!")
files.download('chovus_full_dataset.json')

Sačuvano 1 primera...
Sačuvano 101 primera...
Sačuvano 201 primera...
Sačuvano 301 primera...
Sačuvano 401 primera...
Sačuvano 501 primera...
Sačuvano 601 primera...
Sačuvano 701 primera...
Sačuvano 801 primera...
Sačuvano 901 primera...
Sačuvano 1001 primera...
Sačuvano 1101 primera...
Sačuvano 1201 primera...
Sačuvano 1301 primera...
Sačuvano 1401 primera...
Sačuvano 1501 primera...
Sačuvano 1601 primera...
Sačuvano 1701 primera...
Sačuvano 1801 primera...
Sačuvano 1901 primera...
Sačuvano 2001 primera...
Sačuvano 2101 primera...
Sačuvano 2201 primera...
Sačuvano 2301 primera...
Sačuvano 2401 primera...
Sačuvano 2501 primera...
Sačuvano 2601 primera...
Dataset sačuvan!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from datasets import Dataset

def load_and_fix_dataset(file_path):
    fixed_data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            try:
                data = json.loads(line.strip())

                if "messages" not in data or not isinstance(data["messages"], list):
                    continue

                fixed_messages = []
                valid_conversation = True

                for msg in data["messages"]:
                    if not isinstance(msg, dict) or "role" not in msg:
                        valid_conversation = False
                        break

                    role = msg["role"]

                    # FIX 1: Obavezno content polje
                    if "content" not in msg:
                        valid_conversation = False
                        break

                    content = msg["content"]

                    # Konvertuj content u string ako je objekat
                    if isinstance(content, dict):
                        if role == "assistant" and "text" in content:
                            content = content["text"]
                        else:
                            content = str(content)
                    elif not isinstance(content, str):
                        content = str(content)

                    # FIX 2: Ujednači thinking polje
                    thinking = msg.get("thinking")
                    if thinking is None:
                        thinking = []  # Uvek lista, čak i ako je prazna
                    elif not isinstance(thinking, list):
                        thinking = []  # Ako nije lista, postavi na praznu

                    # Proveri da li su svi thinking elementi validni
                    cleaned_thinking = []
                    if thinking:
                        for thought in thinking:
                            if isinstance(thought, dict) and "channel" in thought and "content" in thought:
                                cleaned_thinking.append({
                                    "channel": str(thought["channel"]),
                                    "content": str(thought["content"])
                                })

                    fixed_msg = {
                        "role": role,
                        "content": content,
                        "thinking": cleaned_thinking  # Uvek lista
                    }

                    fixed_messages.append(fixed_msg)

                # Proveri da li ima bar jedan user i jedan assistant
                has_user = any(msg["role"] == "user" for msg in fixed_messages)
                has_assistant = any(msg["role"] == "assistant" for msg in fixed_messages)

                if valid_conversation and has_user and has_assistant and len(fixed_messages) >= 2:
                    fixed_data.append({
                        "messages": fixed_messages
                    })

            except json.JSONDecodeError as e:
                print(f"Greška u liniji {line_num}: {e}")
                continue
            except Exception as e:
                print(f"Neočekivana greška u liniji {line_num}: {e}")
                continue

    return fixed_data

print("Učitavam i popravljam dataset...")
fixed_dataset = load_and_fix_dataset(dataset_path)

if fixed_dataset:
    print(f"Uspešno popravljeno {len(fixed_dataset)} primera")

    # Kreiraj HuggingFace Dataset
    dataset = Dataset.from_list(fixed_dataset)
    print(dataset)

    # Prikaži strukturu prvog primera
    print("\n--- Struktura popravljenog unosa ---")
    for i, msg in enumerate(dataset[0]["messages"]):
        print(f"Poruka {i}: role={msg['role']}, content_len={len(msg['content'])}, thinking_len={len(msg['thinking'])}")

else:
    print("Nije pronađen nijedan ispravan primer!")

To format our dataset, we will apply our version of the GPT OSS prompt

In [28]:
def formatting_prompts_func(examples):
    texts = []

    for convo in examples["messages"]:
        # Konvertuj thinking liste u string za template
        formatted_convo = []

        for msg in convo:
            if msg["role"] == "assistant" and msg["thinking"]:
                # Kombinuj sve thinking tokene u jedan string
                thinking_text = ""
                for thought in msg["thinking"]:
                    thinking_text += f"[{thought['channel']}] {thought['content']}\n"

                # Kreiraj assistant poruku sa thinking stringom
                formatted_msg = msg.copy()
                formatted_msg["thinking"] = thinking_text.strip()
                formatted_convo.append(formatted_msg)
            else:
                # Za sve ostale poruke, koristi original
                formatted_convo.append(msg)

        # Primeni chat template
        text = tokenizer.apply_chat_template(
            formatted_convo,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

print("Formatiranje dataseta sa thinking stringovima...")
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
print(formatted_dataset)

# Proveri kako izgleda formatiran tekst
print("\n=== FORMATIRANI TEKST (primer) ===")
if len(formatted_dataset) > 0:
    sample_text = formatted_dataset[0]["text"]
    print(sample_text[:1500] + "..." if len(sample_text) > 1500 else sample_text)

Formatiranje dataseta sa thinking stringovima...


Map:   0%|          | 0/2684 [00:00<?, ? examples/s]

TypeError: can only concatenate str (not "list") to str

In [27]:
def manual_gpt_oss_formatting(examples):
    texts = []

    for convo in examples["messages"]:
        formatted_text = ""

        for msg in convo:
            role = msg["role"]
            content = msg["content"]

            if role == "assistant" and msg.get("thinking"):
                # Za assistante sa thinking, kreiraj analysis -> final tokene
                for thought in msg["thinking"]:
                    formatted_text += f"<|start|>assistant<|channel|>{thought['channel']}<|message|>{thought['content']}<|end|>\n"
                # Finalni odgovor
                formatted_text += f"<|start|>assistant<|channel|>final<|message|>{content}<|end|>\n"
            else:
                # Standardne poruke
                formatted_text += f"<|start|>{role}<|message|>{content}<|end|>\n"

        texts.append(formatted_text.strip())

    return {"text": texts}

print("Manualno GPT-OSS formatiranje...")
formatted_dataset = dataset.map(formatting_prompts_func, batched = True)

Manualno GPT-OSS formatiranje...


Map:   0%|          | 0/2684 [00:00<?, ? examples/s]

TypeError: can only concatenate str (not "list") to str

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Let's take a look at the dataset, and check what the 1st example shows

In [25]:
print(dataset[0]['text'])

KeyError: 'text'

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [24]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


RuntimeError: Unsloth: You must specify a `formatting_func`

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

In [ ]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(instruction_part = "<|start|>user<|message|>", response_part="<|start|>assistant<|channel|>final<|message|>")

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
12.811 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
The tokenizer has new special tokens that are also defined in the model configs. The model configs were aligned accordingly. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.130500
2,2.918100
3,2.419300
4,2.167900
5,1.978200
6,2.119900
7,1.825800
8,1.703400
9,1.974400
10,1.796700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

645.6936 seconds used for training.
10.76 minutes used for training.
Peak reserved memory = 12.975 GB.
Peak reserved memory for training = 0.164 GB.
Peak reserved memory % of max memory = 88.02 %.
Peak reserved memory for training % of max memory = 1.113 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The equation is \(x^5 + 3x^4 - 10 = 3\), or \(x^5 + 3x^4 - 13 = 0\). So we need to find the roots of \(x^5 + 3x^4 - 13


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

In [ ]:
model.save_pretrained("finetuned_model")
# model.push_to_hub("hf_username/finetuned_model", token = "hf_...") # Save to HF

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )

messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation for x. The equation: x^5 + 3x^4 - 10 = 3. So bring 3 to left side: x^5 + 3x^4 -10 -3 = 0 → x^5 + 3x^


### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).


In [31]:
# Prikaži prvih 2684 primera detaljno
print("=== PRVIH 5 PRIMERA ===")
for i in range(min(2684, len(dataset))):
    print(f"\n--- Primer {i+1} ---")
    for j, msg in enumerate(dataset[i]["messages"]):
        print(f"Poruka {j}: role={msg['role']}")
        print(f"Content: {msg['content']}...")  # Prvih 200 karaktera
        if msg['thinking']:
            print(f"Thinking ({len(msg['thinking'])} thoughts):")
            for k, thought in enumerate(msg['thinking']):
                print(f"  {k+1}. [{thought['channel']}] {thought['content']}...")
        print("-" * 50)

Streaming output truncated to the last 5000 lines.
Nisu prihvaćeni kao razlog neblagodarnosti npr. obraćanje nadležnim organima u postupcima protiv poklonodavca, kao ni postupci koji se ne mogu kvalifikovati kao „velika“ ili „gruba“ neblagodarnost, već „manja“ ili „sitna. Sudovi nisu kvalifikovali kao grubu neblagodarnost ni odsustvo posebne pažnje, pomoći i zahvalnosti koju je poklonodavac očekivao, a nije dobio od poklonoprimca.
U odgovoru na tužbu navedite sve dokaze koji govore u prilog tome da prema tužilji niste pokazali grubu neblagodarnost.
Razlozi koje je tužilja navela govore u prilog tome da ste se poneli neblagodarno, da li se radi o „gruboj“ neblagodarnosti ceniće sud na osnovu svih okolnosti.
Naravno, ukoliko vi uspete da dokažete da su navodi tužilje neistiniti sud će usvojiti vaš zahtev i odbaciti tužbu.
Sud će ocenu doneti na osnovu raspoloživih dokaza, potrudite se da svoje navode potkrepite sa što više dokaza.
Izjava svedoka koji je tužilji izdavao stan u kome je živ